In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder \
    .appName("ExpenseMonitoringETL") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [2]:
from google.colab import files

uploaded = files.upload()


Saving expenses_cleaned.csv to expenses_cleaned.csv


In [3]:
df = spark.read.csv("expenses_cleaned.csv",
    header=True,
    inferSchema=True
)

print("Expense Dataset")
df.show()

Expense Dataset
+----------+-------+-------------+-------------+------+------------+--------------------+-------------------+
|expense_id|user_id|    user_name|     category|amount|expense_date|         description|              month|
+----------+-------+-------------+-------------+------+------------+--------------------+-------------------+
|         1|      1|Alice Johnson|         Food| 150.0|  2025-04-02|    Grocery shopping|2025-04-01 00:00:00|
|         2|      1|Alice Johnson|    Transport|  45.0|  2025-04-05|    Monthly bus pass|2025-04-01 00:00:00|
|         3|      1|Alice Johnson|    Utilities| 120.0|  2025-04-10|    Electricity bill|2025-04-01 00:00:00|
|         4|      1|Alice Johnson|Entertainment|  30.0|  2025-04-15|Netflix subscription|2025-04-01 00:00:00|
|         5|      1|Alice Johnson|         Food|  80.0|  2025-04-20|   Restaurant dinner|2025-04-01 00:00:00|
|         6|      1|Alice Johnson|       Health|  75.0|  2025-04-28|      Gym membership|2025-04-01 00:0

In [4]:
df.printSchema()

root
 |-- expense_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- month: timestamp (nullable = true)



In [6]:
expense_summary = df.groupBy(
    "user_name",
    "category"
).agg(
    count("*").alias("total_transactions"),
    sum("amount").alias("total_spending"),
    avg("amount").alias("average_spending")
)

print("Expense Summary")
expense_summary.show()

Expense Summary
+-------------+-------------+------------------+--------------+------------------+
|    user_name|     category|total_transactions|total_spending|  average_spending|
+-------------+-------------+------------------+--------------+------------------+
|Alice Johnson|       Health|                 1|          75.0|              75.0|
|    Bob Smith|         Food|                 3|         290.0| 96.66666666666667|
|Alice Johnson|    Transport|                 2|          95.0|              47.5|
|Alice Johnson|    Utilities|                 3|         365.0|121.66666666666667|
|    Bob Smith|    Transport|                 2|         100.0|              50.0|
|Alice Johnson|         Food|                 5|         645.5|             129.1|
|    Bob Smith|       Health|                 2|         250.0|             125.0|
|    Bob Smith|    Utilities|                 1|          90.0|              90.0|
|    Bob Smith|Entertainment|                 2|          40.0|        

In [7]:
alerts_df = df.withColumn(
    "spending_alert",
    when(
        col("amount") > 5000,
        "HIGH SPENDING ALERT"
    ).otherwise("NORMAL")
)

print("Expense Alerts")
alerts_df.show()

Expense Alerts
+----------+-------+-------------+-------------+------+------------+--------------------+-------------------+--------------+
|expense_id|user_id|    user_name|     category|amount|expense_date|         description|              month|spending_alert|
+----------+-------+-------------+-------------+------+------------+--------------------+-------------------+--------------+
|         1|      1|Alice Johnson|         Food| 150.0|  2025-04-02|    Grocery shopping|2025-04-01 00:00:00|        NORMAL|
|         2|      1|Alice Johnson|    Transport|  45.0|  2025-04-05|    Monthly bus pass|2025-04-01 00:00:00|        NORMAL|
|         3|      1|Alice Johnson|    Utilities| 120.0|  2025-04-10|    Electricity bill|2025-04-01 00:00:00|        NORMAL|
|         4|      1|Alice Johnson|Entertainment|  30.0|  2025-04-15|Netflix subscription|2025-04-01 00:00:00|        NORMAL|
|         5|      1|Alice Johnson|         Food|  80.0|  2025-04-20|   Restaurant dinner|2025-04-01 00:00:00| 

In [8]:
monthly_income = 50000

savings_df = expense_summary.withColumn(
    "estimated_savings",
    monthly_income - col("total_spending")
)

print("Savings Report")
savings_df.show()

Savings Report
+-------------+-------------+------------------+--------------+------------------+-----------------+
|    user_name|     category|total_transactions|total_spending|  average_spending|estimated_savings|
+-------------+-------------+------------------+--------------+------------------+-----------------+
|Alice Johnson|       Health|                 1|          75.0|              75.0|          49925.0|
|    Bob Smith|         Food|                 3|         290.0| 96.66666666666667|          49710.0|
|Alice Johnson|    Transport|                 2|          95.0|              47.5|          49905.0|
|Alice Johnson|    Utilities|                 3|         365.0|121.66666666666667|          49635.0|
|    Bob Smith|    Transport|                 2|         100.0|              50.0|          49900.0|
|Alice Johnson|         Food|                 5|         645.5|             129.1|          49354.5|
|    Bob Smith|       Health|                 2|         250.0|             

In [9]:
savings_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/FileStore/output/expense_summary_csv")

print("CSV Report Saved")

CSV Report Saved


In [ ]:
savings_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/FileStore/output/expense_summary_delta")

print("Delta Table Saved")